<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/PytorchCNNFakeVsReal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets,transforms
from torch.utils.data import DataLoader

import os
from google.colab import userdata
from tqdm import tqdm
torch.backends.cudnn.benchmark = True

In [ ]:
print(torch.cuda.is_available())

True


In [ ]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('username and key activated')

username and key activated


In [ ]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:18<00:00, 218MB/s]



In [ ]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

In [ ]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [ ]:
# =========================
# 1. DEVICE (CPU / GPU)
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('usage device: ',device)

usage device:  cuda


In [ ]:
# =========================
# 2. TRANSFORMS
# =========================
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

validation_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [17]:
# =========================
# 3. DATASET + DATALOADER
# =========================
train_dataset = datasets.ImageFolder(train_dir,transform=train_transforms)
validation_dataset = datasets.ImageFolder(validation_dir,transform=validation_transforms)

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=32,
                          shuffle=True,
                          num_workers=2,    # Use 2 or 4 workers in Colab
                          pin_memory=True)   # Speeds up the transfer from CPU to GPU)
validation_loader = DataLoader(dataset=validation_dataset,batch_size=32,num_workers=2,pin_memory=True)

In [18]:
# =========================
# 4. MODEL (CNN)
# =========================
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
# formula for choose padding if wanna set same = kernal -1 / 2
    self.conv = nn.Sequential(
        nn.Conv2d(3,32,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(32,32,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)), # 32 features

        nn.Conv2d(32,64,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(64,64,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)), # 64 features

        nn.Conv2d(64,128,3,padding=1),
        nn.ReLU(),
        nn.Conv2d(128,128,3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=(2,2),stride=(2,2)) # 128 features
    )
    self.flatten = nn.Flatten()

    #Final layer / output layer
    self.fc = nn.Sequential(
        nn.LazyLinear(256),
        nn.ReLU(),
        nn.Linear(256,1)
    )

  def forward(self,x):
    x = self.conv(x)
    x = self.flatten(x)
    x = self.fc(x)
    return x

In [19]:
model = CNN().to(device)

In [20]:
# =========================
# 5. LOSS + OPTIMIZER
# =========================
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(),lr=0.0001)

In [ ]:
# =========================
# 6. TRAINING LOOP
# =========================
epochs = 10

for epoch in range(epochs):
  #--------Train-------#
  model.train()
  train_loss=0
  train_correct=0
  train_total=0


  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.float().unsqueeze(1).to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = loss_fn(outputs,labels)

    loss.backward()
    optimizer.step() # This is for update the weigths based on error. w = oldw - learningrate * grad

    train_loss += loss.item()

    preds = (torch.sigmoid(outputs) > 0.5).int()
    train_correct += (preds.squeeze() == labels.squeeze()).sum().item()
    train_total += labels.size(0)

  train_acc = train_correct / train_total

  # -------- VALIDATION --------
  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for images,labels in validation_loader:
      images = images.to(device)
      labels = labels.float().unsqueeze(1).to(device)

      outputs = model(images)
      loss = loss_fn(outputs,labels)

      val_loss += loss.item()

      preds = (torch.sigmoid(outputs) > 0.5).int()
      val_correct += (preds.squeeze() == labels.squeeze()).sum().item()
      val_total += labels.size(0)

  val_accuracy = val_correct / val_total

  # -------- LOG --------
  print(f"Epoch {epoch+1}/{epochs} "
      f"- loss: {train_loss/train_total:.4f} "
      f"- accuracy: {train_acc:.4f} "
      f"- val_loss: {val_loss/val_total:.4f} "
      f"- val_accuracy: {val_accuracy:.4f}")

Epoch 1/10 - loss: 0.0163 - accuracy: 0.7327 - val_loss: 0.0137 - val_accuracy: 0.8000
Epoch 2/10 - loss: 0.0111 - accuracy: 0.8423 - val_loss: 0.0096 - val_accuracy: 0.8688
Epoch 3/10 - loss: 0.0075 - accuracy: 0.9009 - val_loss: 0.0069 - val_accuracy: 0.9111
Epoch 4/10 - loss: 0.0051 - accuracy: 0.9356 - val_loss: 0.0059 - val_accuracy: 0.9236
Epoch 5/10 - loss: 0.0037 - accuracy: 0.9557 - val_loss: 0.0048 - val_accuracy: 0.9445
Epoch 6/10 - loss: 0.0026 - accuracy: 0.9688 - val_loss: 0.0045 - val_accuracy: 0.9456
Epoch 7/10 - loss: 0.0019 - accuracy: 0.9782 - val_loss: 0.0047 - val_accuracy: 0.9497
Epoch 8/10 - loss: 0.0014 - accuracy: 0.9833 - val_loss: 0.0051 - val_accuracy: 0.9466
Epoch 9/10 - loss: 0.0011 - accuracy: 0.9873 - val_loss: 0.0042 - val_accuracy: 0.9601
